In [1]:
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

In [2]:
def create_data_model():
    """Stores the data for the problem."""
    data = {}

    data["distance_matrix"] = [
        [0,   548, 776, 696, 582, 274, 502],
        [548, 0,   684, 308, 194, 502, 730],
        [776, 684, 0,   992, 878, 502, 274],
        [696, 308, 992, 0,   114, 650, 878],
        [582, 194, 878, 114, 0,   536, 764],
        [274, 502, 502, 650, 536, 0,   228],
        [502, 730, 274, 878, 764, 228, 0  ],
    ]

    # How much of each fuel type each customer needs(demand) 
    data["petrol_demands"]   = [0, 40, 30, 80, 20,  0, 40]
    data["diesel_demands"]   = [0, 60, 60,  0, 80, 40, 20]
    data["kerosene_demands"] = [0,  0, 50, 40,  0, 60, 30]

    # Tank capacity per vehicle for each fuel type
    # 3 vehicles, each with different tank sizes
    data['vehicle_capacities'] = {
        'petrol_capacities' : [100, 100, 80],
        'diesel_capacities' : [150, 120, 100],
        'kerosene_capacities': [80,   80,  60]
    }


    data["num_vehicles"] = 3
    data["depot"] = 0
    return data

In [3]:
data = create_data_model()

# Create the routing index manager
manager = pywrapcp.RoutingIndexManager(
    len(data["distance_matrix"]),data["num_vehicles"],data["depot"],
    )

    # Create routing model
routing = pywrapcp.RoutingModel(manager)

In [4]:
def print_solution(data, manager, routing, solution):
    """Prints solution on console with intuitive load tracking."""
    print(f"Objective: {solution.ObjectiveValue()}")
    print("-" * 50)

    # Get the dimension objects
    petrol_dimension   = routing.GetDimensionOrDie("Petrol")
    diesel_dimension   = routing.GetDimensionOrDie("Diesel")
    kerosene_dimension = routing.GetDimensionOrDie("Kerosene")

    max_route_distance = 0

    for vehicle_id in range(data["num_vehicles"]):
        if not routing.IsVehicleUsed(solution, vehicle_id):
            print(f"Vehicle {vehicle_id}: not used\n")
            continue

        index = routing.Start(vehicle_id)
        plan_output = f"Route for vehicle {vehicle_id}:\n"
        route_distance = 0

        while not routing.IsEnd(index):
            node = manager.IndexToNode(index)

            # 1. Get the load the truck had when it arrived at this stop
            petrol_arrival = solution.Value(petrol_dimension.CumulVar(index))
            diesel_arrival = solution.Value(diesel_dimension.CumulVar(index))
            kerosene_arrival = solution.Value(kerosene_dimension.CumulVar(index))

            # 2. Add the demand of the CURRENT node to show the 'After Pickup' state
            petrol_after = petrol_arrival + data["petrol_demands"][node]
            diesel_after = diesel_arrival + data["diesel_demands"][node]
            kerosene_after = kerosene_arrival + data["kerosene_demands"][node]

            plan_output += (
                f"  Stop {node}"
                f" [P:{petrol_after}L D:{diesel_after}L K:{kerosene_after}L] -> "
            )

            previous_index = index
            index = solution.Value(routing.NextVar(index))
            route_distance += routing.GetArcCostForVehicle(
                previous_index, index, vehicle_id
            )

        # Print the final return to depot
        plan_output += f" Stop {manager.IndexToNode(index)} (depot)\n"
        plan_output += f"  Distance: {route_distance}m\n"

        # Summary: Use routing.End to get the total cumulative load delivered
        end_idx = routing.End(vehicle_id)
        
        plan_output += (
            f"  Petrol capacity used:   "
            f"{solution.Value(petrol_dimension.CumulVar(end_idx))} / "
            f"{data['vehicle_capacities']['petrol_capacities'][vehicle_id]}L\n"
        )
        plan_output += (
            f"  Diesel capacity used:   "
            f"{solution.Value(diesel_dimension.CumulVar(end_idx))} / "
            f"{data['vehicle_capacities']['diesel_capacities'][vehicle_id]}L\n"
        )
        plan_output += (
            f"  Kerosene capacity used: "
            f"{solution.Value(kerosene_dimension.CumulVar(end_idx))} / "
            f"{data['vehicle_capacities']['kerosene_capacities'][vehicle_id]}L\n"
        )

        print(plan_output)
        max_route_distance = max(route_distance, max_route_distance)

    print(f"Maximum route distance: {max_route_distance}m")


In [5]:
# ── Distance callback ──────────────────────────────────────────────────

def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node   = manager.IndexToNode(to_index)
    return data["distance_matrix"][from_node][to_node]

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

In [6]:
# ── Petrol demand callback + dimension ────────────────────────────────
def petrol_callback(from_index, to_index):
     from_node = manager.IndexToNode(from_index)
     return data["petrol_demands"][from_node]

petrol_callback_index = routing.RegisterTransitCallback(petrol_callback)
    
routing.AddDimensionWithVehicleCapacity(
        petrol_callback_index,
        0,                              # no slack
        data['vehicle_capacities']["petrol_capacities"],      # capacity per vehicle
        True,                           # start cumul at zero
        "Petrol",                       # unique name
    )

True

In [7]:
# ── Diesel demand callback + dimension ────────────────────────────────
def diesel_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    return data["diesel_demands"][from_node]

diesel_callback_index = routing.RegisterTransitCallback(diesel_callback)
 
routing.AddDimensionWithVehicleCapacity(
        diesel_callback_index,
        0,
        data['vehicle_capacities']["diesel_capacities"],
        True,
        "Diesel",
    )

True

In [8]:
# ── Kerosene demand callback + dimension ──────────────────────────────
    
def kerosene_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    return data["kerosene_demands"][from_node]

kerosene_callback_index = routing.RegisterTransitCallback(kerosene_callback)

routing.AddDimensionWithVehicleCapacity(
        kerosene_callback_index,
        0,
        data['vehicle_capacities']["kerosene_capacities"],
        True,
        "Kerosene",
    )

True

In [9]:
# ── Search parameters (Phase 1 + Phase 2) ────────────────────────────
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
)
    
search_parameters.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
)
search_parameters.time_limit.seconds = 5
#search_parameters.log_search = True

In [10]:
# ── Solve ─────────────────────────────────────────────────────────────
solution = routing.SolveWithParameters(search_parameters)

In [11]:
if solution:
    print_solution(data, manager, routing, solution)
else:
    print("No solution found!")

Objective: 4268
--------------------------------------------------
Route for vehicle 0:
  Stop 0 [P:0L D:0L K:0L] ->   Stop 4 [P:20L D:80L K:0L] ->   Stop 3 [P:100L D:80L K:40L] ->  Stop 0 (depot)
  Distance: 1392m
  Petrol capacity used:   100 / 100L
  Diesel capacity used:   80 / 150L
  Kerosene capacity used: 40 / 80L

Route for vehicle 1:
  Stop 0 [P:0L D:0L K:0L] ->   Stop 6 [P:40L D:20L K:30L] ->   Stop 2 [P:70L D:80L K:80L] ->  Stop 0 (depot)
  Distance: 1552m
  Petrol capacity used:   70 / 100L
  Diesel capacity used:   80 / 120L
  Kerosene capacity used: 80 / 80L

Route for vehicle 2:
  Stop 0 [P:0L D:0L K:0L] ->   Stop 5 [P:0L D:40L K:60L] ->   Stop 1 [P:40L D:100L K:60L] ->  Stop 0 (depot)
  Distance: 1324m
  Petrol capacity used:   40 / 80L
  Diesel capacity used:   100 / 100L
  Kerosene capacity used: 60 / 60L

Maximum route distance: 1552m
